# Lab Environment Setup (Fabric Admin)

This notebook provisions a training lab for multiple students from a single CSV file.

**Input CSV columns:** `workspace_name`, `user_name`, `role`

**What it does:**
1. Creates one Fabric workspace for each distinct `workspace_name`.
2. Assigns each `user_name` the `role` from the CSV on its `workspace_name`.

**Requirements:**
- Run this as a Fabric **capacity/tenant admin** (or someone allowed to create workspaces and assign the target capacity).
- The identity running the notebook needs permission to add users to the created workspaces.
- Valid `role` values: `Admin`, `Member`, `Contributor`, `Viewer`.

Role assignments are safe to re-run (they update in place). Workspace names are assumed valid and are created without an existing-workspace check.

## 1. Configuration

Set `CAPACITY_ID` before running. Every workspace created by this notebook must be assigned to that Fabric capacity.

In [ ]:
# ------------------------- EDIT THESE -------------------------

# Fabric capacity ID to attach each workspace to. This value is required.
# Find it in the Fabric Admin portal > Capacity settings, or via the Capacities REST API.
CAPACITY_ID = ""

if not CAPACITY_ID.strip():
    raise ValueError("CAPACITY_ID is required. Set it before running the notebook.")

# Path to the roster CSV with columns: workspace_name, user_name, role
# Options:
#   - Notebook resource file uploaded to the 'builtin' folder:  "./builtin/roster.csv"
#   - Lakehouse file (attach a default lakehouse):              "/lakehouse/default/Files/roster.csv"
#   - OneLake / abfss path
CSV_PATH = "./builtin/roster.csv"

# Optional prefix added to every created workspace display name (helps identify a cohort).
WORKSPACE_PREFIX = ""

# Safety switch. Set to False to preview actions without creating anything.
APPLY_CHANGES = True
# --------------------------------------------------------------

print("Capacity ID     :", CAPACITY_ID)
print("CSV path        :", CSV_PATH)
print("Workspace prefix:", WORKSPACE_PREFIX or "(none)")
print("Apply changes   :", APPLY_CHANGES)

## 2. Authentication & REST helpers

Uses `notebookutils.credentials.getToken('pbi')` to obtain an Entra token valid for both the Fabric REST API (`api.fabric.microsoft.com`) and the Power BI REST API (`api.powerbi.com`). Includes retry handling for throttling (HTTP 429).

In [ ]:
import time
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
PBI_API = "https://api.powerbi.com/v1.0/myorg"

VALID_ROLES = {"admin": "Admin", "member": "Member", "contributor": "Contributor", "viewer": "Viewer"}


def _token():
    return notebookutils.credentials.getToken("pbi")


def _headers():
    return {"Authorization": f"Bearer {_token()}", "Content-Type": "application/json"}


def api_request(method, url, json_body=None, max_retries=6):
    """Call a Fabric/Power BI REST endpoint with retry on throttling and transient errors."""
    for attempt in range(max_retries):
        resp = requests.request(method, url, headers=_headers(), json=json_body)
        if resp.status_code == 429 or resp.status_code >= 500:
            wait = int(resp.headers.get("Retry-After", min(2 ** attempt, 30)))
            print(f"  throttled/transient ({resp.status_code}); retrying in {wait}s...")
            time.sleep(wait)
            continue
        return resp
    return resp


def get_paged(url):
    """Return all items from a paginated Fabric list endpoint (handles continuationToken)."""
    items = []
    next_url = url
    while next_url:
        resp = api_request("GET", next_url)
        resp.raise_for_status()
        data = resp.json()
        items.extend(data.get("value", []))
        token = data.get("continuationToken")
        next_url = f"{url}?continuationToken={token}" if token else None
    return items


# Sanity check the token can be acquired.
_ = _token()
print("Token acquired OK.")

## 3. Load and validate the roster CSV

In [ ]:
import pandas as pd

roster = pd.read_csv(CSV_PATH, dtype=str).fillna("")
roster.columns = [c.strip().lower() for c in roster.columns]

required_cols = {"workspace_name", "user_name", "role"}
missing = required_cols - set(roster.columns)
if missing:
    raise ValueError(f"CSV is missing required column(s): {missing}. Found: {list(roster.columns)}")

# Trim whitespace on all relevant fields.
for col in ["workspace_name", "user_name", "role"]:
    roster[col] = roster[col].str.strip()

# Drop empty rows.
roster = roster[(roster["workspace_name"] != "") & (roster["user_name"] != "")].reset_index(drop=True)

# Validate roles.
bad_roles = roster[~roster["role"].str.lower().isin(VALID_ROLES)]
if len(bad_roles):
    raise ValueError(
        "Invalid role value(s) found. Allowed: Admin, Member, Contributor, Viewer.\n"
        + bad_roles[["workspace_name", "user_name", "role"]].to_string(index=False)
    )

# Normalize role to canonical casing.
roster["role"] = roster["role"].str.lower().map(VALID_ROLES)

distinct_workspaces = sorted(roster["workspace_name"].unique())
print(f"Rows: {len(roster)}")
print(f"Distinct workspaces: {len(distinct_workspaces)}")
print(f"Distinct users: {roster['user_name'].nunique()}")
roster.head(10)

## 4. Create workspaces (one per distinct `workspace_name`)

Creates a new workspace for each distinct `workspace_name` in the CSV. Names are assumed valid; no existing-workspace check is performed.

In [ ]:
def create_workspace(display_name):
    body = {"displayName": display_name, "capacityId": CAPACITY_ID}
    resp = api_request("POST", f"{FABRIC_API}/workspaces", json_body=body)
    resp.raise_for_status()
    return resp.json()["id"]


workspace_ids = {}   # workspace_name (from CSV) -> workspace id
create_results = []

for name in distinct_workspaces:
    display_name = f"{WORKSPACE_PREFIX}{name}"
    if not APPLY_CHANGES:
        create_results.append((display_name, "", "would create (dry-run)"))
        continue
    try:
        ws_id = create_workspace(display_name)
        workspace_ids[name] = ws_id
        create_results.append((display_name, ws_id, "created"))
        print(f"Created workspace: {display_name} ({ws_id})")
    except Exception as e:
        create_results.append((display_name, "", f"ERROR: {e}"))
        print(f"FAILED to create {display_name}: {e}")

pd.DataFrame(create_results, columns=["workspace", "id", "status"])

## 5. Assign role helper

Uses the Power BI *Add/Update Group User* API, which accepts an email address directly (no Entra object-ID lookup required). `POST` adds a new user; if the user already exists on the workspace we fall back to `PUT` to update the role.

In [ ]:
def assign_role(workspace_id, email, role):
    """Add or update a user's role on a workspace. Returns a status string."""
    body = {
        "emailAddress": email,
        "identifier": email,
        "principalType": "User",
        "groupUserAccessRight": role,
    }
    url = f"{PBI_API}/groups/{workspace_id}/users"

    resp = api_request("POST", url, json_body=body)
    if resp.status_code in (200, 201):
        return "added"

    # Already a member -> update the role instead.
    text = (resp.text or "").lower()
    if resp.status_code in (400, 409) and ("already" in text or "exist" in text):
        upd = api_request("PUT", url, json_body=body)
        if upd.status_code in (200, 201):
            return "updated"
        return f"ERROR update ({upd.status_code}): {upd.text[:200]}"

    return f"ERROR ({resp.status_code}): {resp.text[:200]}"


print("assign_role() ready.")

## 6. Assign each student their role from the CSV

In [ ]:
assignment_results = []

for row in roster.itertuples(index=False):
    name, email, role = row.workspace_name, row.user_name, row.role
    ws_id = workspace_ids.get(name)
    if not ws_id:
        assignment_results.append((name, email, role, "SKIPPED: workspace not available"))
        continue
    if not APPLY_CHANGES:
        assignment_results.append((name, email, role, "dry-run"))
        continue
    status = assign_role(ws_id, email, role)
    assignment_results.append((name, email, role, status))
    print(f"{name}: {email} ({role}) -> {status}")

assignments_df = pd.DataFrame(assignment_results, columns=["workspace", "user", "role", "status"])
assignments_df